In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Downloading NLTK

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# PARSING FUNCTIONS

In [ ]:
def parse_vtt_captions(file_path):

    captions = []
    current_start = None
    current_end = None

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        # Detecting Timestamp Lines
        if '-->' in line:
            times = line.split(' --> ')
            current_start = times[0].split(' ')[0]
            current_end = times[1].split(' ')[0]

        # Skipping Header/Empty Lines/Metadata
        elif line and not line.startswith('WEBVTT') and not line.startswith('Kind:') and not line.startswith('Language:'):

            clean_text = re.sub(r'<[^>]+>', '', line)

            # Additional cleanup for duplicate timestamp lines that appear as text
            if clean_text.strip() and current_start:
                # specific check to ensure we aren’t adding the timestamp line itself if parsing failed slightly
                if "-->" not in clean_text:
                    captions.append({
                        'start_time': current_start,
                        'end_time': current_end,
                        'original_text': clean_text.strip()
                    })

    return pd.DataFrame(captions)

def parse_comments_txt(file_path):

    comments = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if line:
            comments.append({'comment_text': line})

    return pd.DataFrame(comments)

# Text Cleaning

In [ ]:
def clean_text_pipeline_with_stats(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove special chars
    text = re.sub(r'[^a-z\s]', '', text)

    # 3. Tokenize
    tokens = word_tokenize(text)
    total_tokens = len(tokens)

    # 4. Filter Stopwords & Lemmatize
    valid_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    # Calculate removed count
    removed_count = total_tokens - len(valid_tokens)

    return " ".join(valid_tokens), removed_count

# Loading and Parsing Data

In [ ]:
print("Parsing files...")
df_captions = parse_vtt_captions('/content/drive/MyDrive/captions.vtt')
df_comments = parse_comments_txt('/content/drive/MyDrive/comments.txt')

print(f"Parsed {len(df_captions)} caption lines.")
print(f"Parsed {len(df_comments)} comments.")


Parsing files...
Parsed 2679 caption lines.
Parsed 66 comments.


# Applying Cleaning Pipeline

In [17]:
print("\nCleaning text and calculating stats...")

# Apply to Captions (expanding tuple into two columns)
df_captions[['cleaned_text', 'sw_removed_count']] = df_captions['original_text'].apply(
    lambda x: pd.Series(clean_text_pipeline_with_stats(x))
)

# Apply to Comments
df_comments[['cleaned_text', 'sw_removed_count']] = df_comments['comment_text'].apply(
    lambda x: pd.Series(clean_text_pipeline_with_stats(x))
)

# --- CALCULATE THE TOTAL FOR YOUR REPORT ---
total_sw_removed = df_captions['sw_removed_count'].sum() + df_comments['sw_removed_count'].sum()

print(f"\n TOTAL STOPWORDS REMOVED: {int(total_sw_removed)} ")



Cleaning text and calculating stats...

 TOTAL STOPWORDS REMOVED: 7971 


# Inspecting Results

In [ ]:
print("\n--- Captions Head ---")
print(df_captions[['original_text', 'cleaned_text']].head())

print("\n--- Comments Head ---")
print(df_comments[['comment_text', 'cleaned_text']].head())


--- Captions Head ---
                          original_text              cleaned_text
0  And now the snakes are on the alert.               snake alert
1  And now the snakes are on the alert.               snake alert
2  And now the snakes are on the alert.               snake alert
3  This is the best feeding opportunity  best feeding opportunity
4  This is the best feeding opportunity  best feeding opportunity

--- Comments Head ---
                                        comment_text  \
0  That bison knocking down the young one like "S...   
1  Therapy is expensive. So, I listen to Sir Davi...   
2  I never get tired of hearing Sir David Attenbo...   
3  Who knew a tiny mantis could channel its inner...   
4  Those snakes after the iguanas is like hell on...   

                                        cleaned_text  
0  bison knocking young one like sorry lil homie ...  
1    therapy expensive listen sir david attenborough  
2  never get tired hearing sir david attenborough...  
3

# Exporting to CSV

In [ ]:
df_captions.to_csv('cleaned_captions.csv', index=False)
df_comments.to_csv('cleaned_comments.csv', index=False)

print("\nFiles saved as 'cleaned_captions.csv' and 'cleaned_comments.csv'.")


Files saved as 'cleaned_captions.csv' and 'cleaned_comments.csv'.


# Saving to Google Drive

In [ ]:
from google.colab import drive
import os
save_folder = '/content/drive/My Drive/Lab2_477'
os.makedirs(save_folder, exist_ok=True)
# Saving the DataFrames to that folder
df_captions.to_csv(os.path.join(save_folder, 'cleaned_captions.csv'), index=False)
df_comments.to_csv(os.path.join(save_folder, 'cleaned_comments.csv'), index=False)

print(f"\nFiles have been saved to Google Drive inside the folder: '{save_folder}'")


Files have been saved to Google Drive inside the folder: '/content/drive/My Drive/Lab2_477'
